In [78]:
import time
from functools import wraps
from io import StringIO

## 5.2 Default Arguments

See [Mutable default argument](../hitchhikers_guide/gotchas.ipynb)

In [2]:
def func(x, items=[]):
    items.append(x)
    return items

print(func(1))      # returns [1]
print(func(2))      # returns [1, 2] because the default list is shared
print(func(3, []))  # returns [3] because a new list is provided

[1]
[1, 2]
[3]


In [39]:
original = [1, 2, 2]


def func_same(x, items=None):
    if items is None:
        items = []
    items.append(x)
    return items


def func_copy(x, items=None):
    items = [] if items is None else items.copy()
    items.append(x)
    return items


print(original)
res1 = func_same(1, original)
print(res1)
print(res1 is original)  # True

res2 = func_copy(4, original)
print(res2)
print(res2 is original)  # False

[1, 2, 2]
[1, 2, 2, 1]
True
[1, 2, 2, 1, 4]
False


## 5.3 Variadic Arguments

In [ ]:
# extra arguments are placed into the args variable as a tuple
def product(first, *args):
    result = first
    for arg in args:
        result += arg
    return result

print(product(10, 20))         # returns 30
print(product(2, 5, 5, 8))     # returns 20

30
20


## 5.4 Keyword Arguments

In [ ]:
def read_data(filename, *, debug=False):
    pass


read_data("file.csv")                # Valid
read_data("file.csv", debug=True)    # Valid
# read_data("file.csv", True)        # TypeError

# after *, scale has to be passed by keyword
# because we wouldn't know how many would be in values
def product(first, *values, scale=1):
    pass


product(2)                  # Valid, values = ()
product(2, 3, 4)            # Valid, values = (3, 4)
product(2, 3, 4, scale=10)  # Valid
# product()                 # TypeError, missing first

## 5.6 Functions Accepting All Inputs

In [ ]:
# this combined use of *args and **kwargs is commonly used to write
# wrappers, decorators, and similiar functions
def parse_lines(lines, *args, **kwargs):
    for line in lines:
        print(line.strip(), args, kwargs)


def parse_file(file, *args, **kwargs):
    return parse_lines(file, *args, **kwargs)


file = StringIO("Hello\nWorld\nPython")

parse_file(file, "arg1", "arg2", debug=True, verbose=False)

# The benefit of this approach is that the parse_file() function
# doesn’t need to know anything about the arguments of parse_lines()


Hello ('arg1', 'arg2') {'debug': True, 'verbose': False}
World ('arg1', 'arg2') {'debug': True, 'verbose': False}
Python ('arg1', 'arg2') {'debug': True, 'verbose': False}


## 5.7 Positional-Only Arguments

In [58]:
# Python 3.8 
def func(x, y, /, z):
    pass

func(1, 2, z=5)  # Valid
# func(x=1, y=2)  # TypeError, positional-only arguments cannot

## 5.9 Function Application and Parameter Passing

In [ ]:
def square(items):
    for i, x in enumerate(items):
        items[i] = x * x


a = [2, 4, 8, 16]
square(a)
print(a)  # [4, 16, 64, 256]

[4, 16, 64, 256]


In [23]:
def sum_squares(items):
    items = sum(x * x for x in items)
    return items


a = [1, 2, 3, 4]
result = sum_squares(a)
print(a)  # [1, 2, 3, 4]
print(result)  # 30

[1, 2, 3, 4]
30


In [26]:
# Stylistically, it is common for functions with side effects to return None as a result.
items = [10, 1, 9, 3]
items.sort() # no return value
items

[1, 3, 9, 10]

## 5.10 Return Values

In [35]:
def parse_value(text):
    '''
    Split text of the form name=val into (name, val)
    '''
    parts = text.split("=", 1)
    return (parts[0].strip(), parts[1].strip())

name, value = parse_value("url=http://www.python.org")
print(name, value)

url http://www.python.org


In [ ]:
from typing import NamedTuple

class ParseResult(NamedTuple):
    name: str
    value: str

def parse_value(text):
    parts = text.split("=", 1)
    return ParseResult(parts[0].strip(), parts[1].strip())

r = parse_value("url=http://www.python.org")
print(r.name, r.value)

url http://www.python.org


## 5.11 Error Handling

In [47]:
# Exceptions are more common way to handle abnormal result
# but they are more expensive if they frequently occur

text="url=http://www.python.org"
def parse_value(text):
    parts = text.split("=", 1)
    if len(parts) == 2:
        return ParseResult(parts[0].strip(), parts[1].strip())
    else:
        return None

result = parse_value(text)
if result:
    name, value = result

# Python 3.8+
if result := parse_value(text):
    name, value = result

def parse_value_raise(text):
    parts = text.split("=", 1)
    if len(parts) == 2:
        return ParseResult(parts[0].strip(), parts[1].strip())
    else:
        raise ValueError("Bad value")

# r = parse_value_raise("test") # ValueError: Bad value

try:
    name, value = parse_value_raise(text)
    ...
except ValueError:
    ...

## 5.12 Scoping Rules

## 5.14 The lambda Expression

## 5.15 Higher-Order Functions

## 5.16 Argument Passing in Callback Functions

## 5.18 Decorators

In [ ]:
def decorate(func):
    return func


# Syntax 1: Using @
@decorate
def func1(x): ...


# Syntax 2: Equivalent without @
def func2(x): ...


func2 = decorate(func2)

In [ ]:
def trace(func):
    @wraps(func)
    def call(*args, **kwargs):
        print("Calling", func.__name__)
        return func(*args, **kwargs)

    return call


def square(x: int) -> int:
    return x * x


# Original function
print(square(5))

print()

# Decorated function
traced_square = trace(square)
print(traced_square(5))

print()


# Equivalent to @trace
@trace
def cube(x: int) -> int:
    return x**3


print(cube(5))

print()

print(traced_square.__name__)  # square
print(traced_square.__wrapped__ is square)  # True

25

Calling square
25

Calling cube
125

square
True


In [70]:
def decorate(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Before function")
        result = func(*args, **kwargs)
        print("After function")
        return result
    return wrapper

@decorate
def greet(name: str) -> None:
    print(f"Hello, {name}!")

greet("David")

Before function
Hello, David!
After function


In [81]:
def timing(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{func.__name__} took {elapsed:.6f} seconds")

    return wrapper


@timing
def slow_function(seconds: float) -> None:
    time.sleep(seconds)


slow_function(2)

slow_function took 2.000240 seconds
